# 10 · Dateien lesen und schreiben

**Lernziele**

- Du arbeitest mit Pfaden über `pathlib`.
- Du liest und schreibst Textdateien mit `with open(...)`.
- Du verarbeitest CSV-Dateien mit dem Modul `csv`.
- Du speicherst und lädst Daten als JSON.

> Grundlage: Inden, *Einstieg in Python*, Kapitel 7 und 14.

## Pfade mit `pathlib`

`Path` beschreibt einen Datei- oder Ordnerpfad, unabhängig vom Betriebssystem. Mit `/` setzt du Pfade zusammen.

Ein relativer Pfad wie `../daten` gilt ab dem **Arbeitsordner**. In Jupyter ist das der Ordner des Notebooks, hier `notebooks/`; `..` führt eine Ebene höher in den Projektordner, dort liegt `daten/`.

![Der relative Pfad ../daten/einkaufsliste.txt Teil für Teil: Vom Arbeitsordner notebooks/ führt .. in den Projektordner, daten in den Ordner daten/ und einkaufsliste.txt zur Datei.](../assets/10_relativer_pfad.png)

In [ ]:
from pathlib import Path

data_dir = Path("../daten")
shopping_list = data_dir / "einkaufsliste.txt"

print(shopping_list)
print(shopping_list.exists())
print(shopping_list.name, shopping_list.suffix)
print(shopping_list.resolve())  # absoluter Pfad

Ordnerinhalt auflisten:

In [ ]:
for path in sorted(data_dir.iterdir()):
    kind = "[DIR]" if path.is_dir() else f"{path.stat().st_size} Bytes"
    print(f"{path.name:<25} {kind}")

**Kurzübung 1 (Datei nicht gefunden):** Die Zelle darunter wirft `FileNotFoundError`, obwohl die Datei `faust.txt` existiert. Finde mit `Path.cwd()` heraus, wo das Notebook sucht, und baue den Pfad so, dass `exists()` `True` liefert.

In [ ]:
with open("daten/faust.txt", encoding="utf-8") as file:
    print(file.readline())

In [ ]:
print(Path.cwd())  # der Ordner dieses Notebooks
faust = Path("../daten/faust.txt")
print(faust.exists(), faust.resolve())
# Relative Pfade gelten ab dem Arbeitsordner, nicht ab dem Projektordner.
# Auch eine Chat-KI kennt den Arbeitsordner des Notebooks nicht; genau daher kommt der Fehler oft.

## Textdateien lesen

`with open(...)` öffnet die Datei und schließt sie am Ende des Blocks automatisch, auch bei Fehlern. Gib immer `encoding="utf-8"` an. Seit Python 3.15 ist UTF-8 zwar überall die Voreinstellung, ältere Versionen nehmen unter Windows aber eine andere Kodierung, und dann gehen Umlaute kaputt. *Explicit is better than implicit:* Mit `encoding="utf-8"` läuft dein Code mit jeder Version gleich.

In [ ]:
with open(shopping_list, encoding="utf-8") as file:
    content = file.read()
print(content)

Was `with` dabei Schritt für Schritt tut:

![Ablauf des with-Blocks: Datei öffnen, content = file.read() ausführen; ohne Fehler wird die Datei automatisch geschlossen und danach print(content) ausgeführt, bei einem Fehler wird sie ebenfalls automatisch geschlossen und der Fehler weitergemeldet.](../assets/10_with_block.png)

Zeile für Zeile lesen ist bei großen Dateien sparsamer:

In [ ]:
with open(shopping_list, encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        print(f"{line_number}: {line.strip()}")

Für kleine Dateien gibt es eine Abkürzung:

In [ ]:
shopping_list.read_text(encoding="utf-8").splitlines()

**Kurzübung 2 (Zeilenenden):** Entferne in der nummerierten Ausgabe oben `.strip()`. Welche Ausgabe erzeugt die Schleife dann, und warum? Überlege erst, dann probiere es aus. Wie bekommst du die kompakte Ausgabe auch ohne `.strip()` zurück?

In [ ]:
with open(shopping_list, encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        print(f"{line_number}: {line}")
# Jede gelesene Zeile endet mit "\n", print hängt ein zweites an: Zwischen den
# Einträgen steht eine Leerzeile. Abhilfe: line.strip() entfernt das "\n" (und
# Leerzeichen am Rand), print(..., end="") lässt die Zeile unverändert:
with open(shopping_list, encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        print(f"{line_number}: {line}", end="")

## Textdateien schreiben

Der **Modus** bestimmt, was passiert:

| Modus | Bedeutung |
| --- | --- |
| `"r"` | lesen (Standard) |
| `"w"` | schreiben, **überschreibt** vorhandenen Inhalt |
| `"a"` | anhängen |

In [ ]:
output_dir = Path("../daten/ausgabe")
output_dir.mkdir(exist_ok=True)
notes = output_dir / "notizen.txt"

with open(notes, "w", encoding="utf-8") as file:
    file.write("Erste Zeile\n")
    file.write("Zweite Zeile\n")

with open(notes, "a", encoding="utf-8") as file:
    file.write("Angehängte Zeile\n")

print(notes.read_text(encoding="utf-8"))

Auch beim Schreiben gibt es für kleine Dateien eine Abkürzung: `write_text` schreibt den ganzen Text auf einmal und überschreibt dabei vorhandenen Inhalt, wie Modus `"w"`.

Willst du aus einer Datei lesen und in eine andere schreiben, öffnest du beide in **einem** `with`, durch Komma getrennt:

In [ ]:
greeting = output_dir / "gruss.txt"
greeting.write_text("Hallo\nTschüss\n", encoding="utf-8")

loud = output_dir / "notizen_gross.txt"
with open(notes, encoding="utf-8") as source, open(loud, "w", encoding="utf-8") as target:
    for line in source:
        target.write(line.upper())

print(greeting.read_text(encoding="utf-8"))
print(loud.read_text(encoding="utf-8"))

**Kurzübung 3 (Nur die letzte Notiz bleibt):** Der Code darunter soll drei Erinnerungen in `erinnerungen.txt` schreiben, in der Datei steht danach aber nur die letzte. Finde den Fehler und behebe ihn. Es gibt zwei Wege: Welchen nimmst du, und warum?

In [ ]:
reminders = ["Milch kaufen", "Zahnarzt anrufen", "Steuer abgeben"]
target = Path("../daten/ausgabe/erinnerungen.txt")
for reminder in reminders:
    with open(target, "w", encoding="utf-8") as file:
        file.write(reminder + "\n")
print(target.read_text(encoding="utf-8"))

In [ ]:
reminders = ["Milch kaufen", "Zahnarzt anrufen", "Steuer abgeben"]
target = Path("../daten/ausgabe/erinnerungen.txt")
with open(target, "w", encoding="utf-8") as file:  # einmal öffnen, nicht pro Erinnerung
    for reminder in reminders:
        file.write(reminder + "\n")
print(target.read_text(encoding="utf-8"))

# Zweiter Weg: in der Schleife mit Modus "a" anhängen. Dann wächst die Datei aber
# bei jedem erneuten Ausführen der Zelle um drei Zeilen. Einmal mit "w" öffnen ist
# hier richtig: Die Datei soll genau diese drei Erinnerungen enthalten.

## CSV-Dateien

CSV ist das Austauschformat für Tabellen, etwa aus Excel. Das Modul `csv` kümmert sich um Trennzeichen und Anführungszeichen. `DictReader` liefert jede Zeile als Dictionary mit den Spaltennamen als Schlüssel.

Öffne CSV-Dateien immer mit `newline=""`: Dann regelt das Modul `csv` die Zeilenenden selbst, und unter Windows entstehen beim Schreiben keine leeren Zwischenzeilen.

In [ ]:
import csv

with open(data_dir / "highscores.csv", encoding="utf-8", newline="") as file:
    for row in csv.DictReader(file):
        print(f"{row['name']:<10} {row['points']:>5} Punkte, Level {row['level']}")

> Werte aus CSV sind immer Strings. Zum Rechnen umwandeln: `int(row["points"])`.

Schreiben mit `DictWriter`:

In [ ]:
results = [
    {"name": "Ada", "points": 1200},
    {"name": "Linus", "points": 950},
]

with open(output_dir / "ergebnisse.csv", "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["name", "points"])
    writer.writeheader()
    writer.writerows(results)

print((output_dir / "ergebnisse.csv").read_text(encoding="utf-8"))

> **Excel und das BOM:** Excel unter Windows erkennt UTF-8 nur sicher, wenn die Datei mit einem unsichtbaren Kennzeichen beginnt, dem *BOM* (Byte Order Mark). Schreibst du für Excel, nimm `encoding="utf-8-sig"`; dann erscheinen Umlaute richtig. Liest du eine Datei aus Excel oder dem Online-Banking, nimm ebenfalls `"utf-8-sig"`. Mit `"utf-8"` bliebe das BOM als unsichtbares Zeichen vor dem ersten Spaltennamen stehen, und `row["Datum"]` fände die Spalte nicht (`KeyError`).

**Kurzübung 4 (Punkte zusammenzählen):** Berechne mit einer eigenen Schleife die Summe aller Punkte in `highscores.csv`. Welcher Fehler entsteht, wenn du `int()` vergisst?

In [ ]:
total = 0
with open(data_dir / "highscores.csv", encoding="utf-8", newline="") as file:
    for row in csv.DictReader(file):
        total += int(row["points"])
print(total)  # 4635

# Ohne int(): TypeError, denn 0 + "1000" verbindet eine Zahl mit einem String.
# Kürzer mit sum() und einem Generator-Ausdruck:
with open(data_dir / "highscores.csv", encoding="utf-8", newline="") as file:
    print(sum(int(row["points"]) for row in csv.DictReader(file)))

## JSON

JSON speichert verschachtelte Daten aus Listen und Dictionaries. Es ist das Standardformat von Web-APIs.

In [ ]:
import json

with open(data_dir / "personen.json", encoding="utf-8") as file:
    data = json.load(file)

for person in data["people"]:
    print(f"{person['name']} aus {person['city']}")

Zum Speichern dient `json.dump`. `indent=2` rückt die Datei lesbar ein, sonst steht alles in einer Zeile. `ensure_ascii=False` schreibt Umlaute als Umlaute, sonst steht ein Ersatzcode in der Datei. Die letzte Zeile zeigt das mit `json.dumps`, das den JSON-Text als String liefert statt ihn in eine Datei zu schreiben:

In [ ]:
data["people"].append({"name": "Grace", "city": "New York"})

with open(output_dir / "personen_neu.json", "w", encoding="utf-8") as file:
    json.dump(data, file, indent=2, ensure_ascii=False)

print((output_dir / "personen_neu.json").read_text(encoding="utf-8"))
print(json.dumps({"user": "Jörg"}))  # ohne ensure_ascii=False

**Kurzübung 5 (JSON-Rundreise):** Speichere die Einstellungen aus `settings` mit `json.dump` in die Datei `../daten/ausgabe/einstellungen.json`. Lies die Datei danach mit `json.load` wieder ein und prüfe mit `==`, ob du dasselbe Dictionary zurückbekommst.

- **Untersuchen:** Mach aus der Liste `recent_files` ein Tupel und wiederhole die Rundreise. Ist das Geladene noch gleich dem Original, und warum? Was passiert, wenn `recent_files` ein Set ist?

In [ ]:
settings = {
    "user": "Jörg",
    "font_size": 14,
    "dark_mode": True,
    "recent_files": ["bericht.docx", "umsatz.csv"],
}

In [ ]:
settings_file = output_dir / "einstellungen.json"

with open(settings_file, "w", encoding="utf-8") as file:
    json.dump(settings, file, indent=2, ensure_ascii=False)

with open(settings_file, encoding="utf-8") as file:
    loaded = json.load(file)

print(loaded)
print(loaded == settings)  # True

# Untersuchen: JSON kennt nur dict, list, str, int, float, bool und None.
# Ein Tupel speichert json.dump als Liste, zurück kommt also eine Liste.
with_tuple = settings.copy()
with_tuple["recent_files"] = ("bericht.docx", "umsatz.csv")
with open(settings_file, "w", encoding="utf-8") as file:
    json.dump(with_tuple, file, indent=2, ensure_ascii=False)
with open(settings_file, encoding="utf-8") as file:
    loaded = json.load(file)
print(loaded["recent_files"], loaded == with_tuple)  # Liste, False

# Ein Set kennt JSON gar nicht: TypeError (Zelle darunter).
# Abhilfe: das Set vor dem Speichern mit sorted(...) in eine Liste umwandeln.

In [ ]:
with_set = settings.copy()
with_set["recent_files"] = {"bericht.docx", "umsatz.csv"}
json.dumps(with_set)  # wie json.dump, liefert den Text aber ohne Datei: kein halb geschriebenes JSON

## Übungen

Die Übungen 1 bis 3 schreiben ihre Ergebnisse nach `../daten/ausgabe/`. Die Übungen 4 bis 7 arbeiten mit Beispieldateien; die jeweilige Vorbereitungszelle legt sie dort an, und du kannst sie beliebig oft ausführen.

**Übung 1 (Checkliste zum Abhaken, leicht):** Mach aus `../daten/einkaufsliste.txt` eine Checkliste zum Ausdrucken: Schreibe jeden Eintrag als `[ ] Milch` in eine eigene Zeile der Datei `../daten/ausgabe/checkliste.txt` und gib die Datei danach zur Kontrolle aus. Schreibe die Datei einmal mit einer Schleife und `write`, einmal mit `write_text`. Welche Fassung nimmst du, und warum?

In [ ]:
checklist = Path("../daten/ausgabe/checkliste.txt")
items = Path("../daten/einkaufsliste.txt").read_text(encoding="utf-8").splitlines()

# Fassung 1: Schleife mit write
with open(checklist, "w", encoding="utf-8") as file:
    for item in items:
        file.write(f"[ ] {item}\n")
print(checklist.read_text(encoding="utf-8"))

# Fassung 2: den ganzen Text auf einmal bauen und mit write_text schreiben
checklist.write_text("".join(f"[ ] {item}\n" for item in items), encoding="utf-8")
print(checklist.read_text(encoding="utf-8"))

# Bei kleinen Dateien sind beide gleich gut; Fassung 2 ist kürzer. Fassung 1 bleibt
# übersichtlich, wenn pro Zeile mehr passiert, etwa Leerzeilen überspringen.

**Übung 2 (Teilnehmerliste für Excel, leicht):** Ein deutsches Excel erwartet in CSV-Dateien ein Semikolon als Trennzeichen. Schreibe die Teilnehmenden aus `participants` mit `csv.DictWriter` und `delimiter=";"` samt Kopfzeile in die Datei `../daten/ausgabe/teilnehmer.csv` und gib die Datei danach zur Kontrolle aus.

- **Untersuchen:** Ein Hinweis enthält selbst ein Semikolon. Wie schreibt das Modul `csv` ihn in die Datei? Was würde stattdessen `";".join(person.values())` daraus machen?

In [ ]:
participants = [
    {"name": "Anna Becker", "email": "anna.becker@example.org", "hinweis": ""},
    {"name": "Jörg Schmitt", "email": "j.schmitt@example.org", "hinweis": "vegetarisch; kommt 30 min später"},
    {"name": "Lena Yilmaz", "email": "lena.yilmaz@example.org", "hinweis": "Parkplatz benötigt"},
]

In [ ]:
participants_file = Path("../daten/ausgabe/teilnehmer.csv")

with open(participants_file, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["name", "email", "hinweis"], delimiter=";")
    writer.writeheader()
    writer.writerows(participants)

print(participants_file.read_text(encoding="utf-8"))

# Untersuchen: csv setzt den Hinweis mit Semikolon in Anführungszeichen,
# "vegetarisch; kommt 30 min später", und Excel liest ihn als eine Zelle.
# ";".join(...) setzt keine Anführungszeichen: Die Zeile hätte vier Spalten statt drei.
print(";".join(participants[1].values()))

# Praxistipp: Excel unter Windows erkennt UTF-8 nur mit BOM sicher. Mit
# encoding="utf-8-sig" erscheinen die Umlaute in "Jörg" auch dort richtig.

**Übung 3 (Aufgabenliste, die sich alles merkt, leicht):** Deine Aufgabenliste soll das Schließen des Notebooks überstehen. Lade die Liste aus `../daten/ausgabe/aufgaben.json`, falls die Datei existiert, sonst beginne mit einer leeren Liste. Hänge die Aufgabe aus `new_task` an, speichere die Liste wieder als JSON und gib alle Aufgaben nummeriert aus. Führe die Zellen mehrmals mit verschiedenen Aufgaben aus: Die Liste wächst.

- **Untersuchen:** Was passiert, wenn `aufgaben.json` zwar existiert, aber leer ist?

In [ ]:
new_task = "Rechnung an Kunde Meier schicken"

In [ ]:
tasks_file = Path("../daten/ausgabe/aufgaben.json")

if tasks_file.exists():
    with open(tasks_file, encoding="utf-8") as file:
        tasks = json.load(file)
else:
    tasks = []

tasks.append(new_task)

with open(tasks_file, "w", encoding="utf-8") as file:
    json.dump(tasks, file, indent=2, ensure_ascii=False)

for number, task in enumerate(tasks, start=1):
    print(f"{number}. {task}")

# Untersuchen: Eine leere Datei ist kein gültiges JSON. json.load wirft dann einen
# json.JSONDecodeError (eine Unterart von ValueError); abfangen mit try/except (Notebook 09).

**Übung 4 (Logdatei nach Fehlern durchsuchen, mittel):** Lies `../daten/ausgabe/beispieldaten/server.log` (die Zelle darunter erzeugt sie), zähle die Zeilen je Level und schreibe alle `ERROR`-Zeilen mit Zeilennummer nach `../daten/ausgabe/fehler.txt`. Öffne beide Dateien in einem `with`.

Tipp: `from collections import Counter`; ein `Counter` zählt gleiche Einträge (wie in Notebook 06 bei den Logzeilen), `.most_common()` sortiert nach Häufigkeit.

In [ ]:
# Vorbereitung: erzeugt eine Beispiel-Logdatei (Datum, Uhrzeit, Level, Meldung)
sample_dir = Path("../daten/ausgabe/beispieldaten")
sample_dir.mkdir(parents=True, exist_ok=True)
log_text = """\
2026-11-02 08:58:01 INFO Server gestartet auf Port 8080
2026-11-02 08:58:02 INFO Datenbankverbindung hergestellt
2026-11-02 09:00:12 INFO Anmeldung: user=anna
2026-11-02 09:01:45 INFO Bericht erstellt: umsatz_oktober.pdf
2026-11-02 09:03:10 WARNING Langsame Antwort: /kunden (2.4 s)
2026-11-02 09:05:33 INFO Anmeldung: user=ben
2026-11-02 09:07:02 ERROR Datei nicht gefunden: rechnung_4711.pdf
2026-11-02 09:07:03 INFO Anmeldung: user=chris
2026-11-02 09:10:27 WARNING Speicherplatz unter 15 %
2026-11-02 09:12:48 INFO Bericht erstellt: kunden.csv
2026-11-02 09:15:00 INFO Sicherung gestartet
2026-11-02 09:18:21 ERROR Sicherung abgebrochen: Zielordner nicht erreichbar
2026-11-02 09:18:22 INFO Sicherung wird in 5 Minuten wiederholt
2026-11-02 09:20:05 INFO Abmeldung: user=anna
2026-11-02 09:23:22 INFO Sicherung gestartet
2026-11-02 09:26:40 INFO Sicherung abgeschlossen
2026-11-02 09:30:11 WARNING Falsches Passwort: user=dora
2026-11-02 09:30:19 WARNING Falsches Passwort: user=dora
2026-11-02 09:30:25 ERROR Konto gesperrt nach 3 Fehlversuchen: user=dora
2026-11-02 09:35:50 INFO Anmeldung: user=emil
2026-11-02 09:41:02 INFO Bericht erstellt: lager.xlsx
2026-11-02 09:44:17 WARNING Langsame Antwort: /lager (3.1 s)
2026-11-02 09:48:39 INFO Abmeldung: user=ben
2026-11-02 09:52:08 ERROR Datenbankverbindung verloren
2026-11-02 09:52:10 INFO Datenbankverbindung hergestellt
2026-11-02 09:55:31 INFO Abmeldung: user=chris
2026-11-02 10:00:00 INFO Stündlicher Statusbericht: alles in Ordnung
2026-11-02 10:02:44 INFO Anmeldung: user=anna
2026-11-02 10:05:16 WARNING Zertifikat läuft in 14 Tagen ab
2026-11-02 10:09:58 ERROR Mailversand fehlgeschlagen: smtp.example.org
"""
(sample_dir / "server.log").write_text(log_text, encoding="utf-8")
print("angelegt:", sample_dir / "server.log")

In [ ]:
from collections import Counter

log = Path("../daten/ausgabe/beispieldaten/server.log")
errors = Path("../daten/ausgabe/fehler.txt")
levels = Counter()
with open(log, encoding="utf-8") as source, open(errors, "w", encoding="utf-8") as target:
    for number, line in enumerate(source, start=1):
        day, clock, level, message = line.split(maxsplit=3)
        levels[level] += 1
        if level == "ERROR":
            target.write(f"{number}: {line}")  # line endet schon mit "\n"
print(levels.most_common())
print(errors.read_text(encoding="utf-8"))

**Übung 5 (Kontoauszug: Annahmen brechen, mittel):** Der Startcode (wie ihn eine KI für „Beträge aus dem Online-Banking summieren“ schreiben könnte) läuft mit den Beispielwerten. Welche Annahmen trifft er über die Beträge? Schreibe für jede Annahme einen Wert, der sie verletzt, und prüfe ihn.

Werte dann `../daten/ausgabe/beispieldaten/kontoauszug.csv` robust aus (die Vorbereitungszelle erzeugt sie: Semikolon, Dezimalkomma, Tausenderpunkt, BOM). Entferne den Tausenderpunkt, überspringe Zeilen ohne Betrag und gib Einnahmen, Ausgaben und Saldo aus. Erwartet: `Einnahmen 2812.40 €, Ausgaben 2146.25 €, Saldo 666.15 €`.

Tipp: Öffne die Datei mit `encoding="utf-8-sig"` und lies sie mit `csv.DictReader(file, delimiter=";")`.

- **Untersuchen:** Was passiert mit `"2.500,00"` (Tausenderpunkt) und mit `""` (leerer Betrag)? Bricht eine leere Zeile am Ende der CSV-Datei die Auswertung?

In [ ]:
from decimal import Decimal

amounts = ["-850,00", "-62,35", "1200,00"]  # aus dem Kontoauszug
total = Decimal(0)
for amount in amounts:
    total += Decimal(amount.replace(",", "."))
print("Saldo:", total, "€")

In [ ]:
# Vorbereitung: erzeugt einen Kontoauszug, wie ihn deutsche Banken exportieren
# (Semikolon, Dezimalkomma, Tausenderpunkt, BOM, eine Zeile ";;;", Leerzeile am Ende).
sample_dir = Path("../daten/ausgabe/beispieldaten")
sample_dir.mkdir(parents=True, exist_ok=True)
statement_text = """\
Datum;Empfänger;Verwendungszweck;Betrag
01.10.2026;Arbeitgeber GmbH;Gehalt Oktober;2.500,00
01.10.2026;Hausverwaltung Meier;Miete Oktober;-850,00
02.10.2026;Stadtwerke;Abschlag Strom;-62,35
05.10.2026;Supermarkt;Einkauf;-45,10
12.10.2026;Supermarkt;Einkauf;-38,90
15.10.2026;Fitnessstudio;Beitrag;-29,90
20.10.2026;Hausverwaltung Meier;Nebenkosten 2025;-1.120,00
;;;
28.10.2026;Finanzamt;Erstattung 2025;312,40

"""
(sample_dir / "kontoauszug.csv").write_text(statement_text, encoding="utf-8-sig")
print("angelegt:", sample_dir / "kontoauszug.csv")

In [ ]:
from decimal import InvalidOperation

# Annahmen des Startcodes: Komma als Dezimaltrenner, kein Tausenderpunkt, nie leer.
for breaking in ["2.500,00", ""]:
    try:
        Decimal(breaking.replace(",", "."))
    except InvalidOperation:
        print(f"bricht den Startcode: {breaking!r}")

# Eine leere Zeile am Dateiende überspringt csv.DictReader; die Zeile ";;;" dagegen
# liefert einen leeren Betrag und bricht die Auswertung ohne eigene Prüfung.
statement = Path("../daten/ausgabe/beispieldaten/kontoauszug.csv")
with open(statement, encoding="utf-8-sig", newline="") as file:
    rows = list(csv.DictReader(file, delimiter=";"))
income, spending = Decimal(0), Decimal(0)
for row in rows:
    if not row["Betrag"]:
        continue  # Zeile ohne Betrag, z. B. ";;;"
    amount = Decimal(row["Betrag"].replace(".", "").replace(",", "."))
    if amount >= 0:
        income += amount
    else:
        spending -= amount
print(f"Einnahmen {income} €, Ausgaben {spending} €, Saldo {income - spending} €")

**Übung 6 (Download-Ordner aufräumen, schwer):** Sortiere alle Dateien in `../daten/ausgabe/downloads` nach Endung in Unterordner (`pdf`, `jpg`, …). Gib zuerst nur aus, was passieren würde (`DRY_RUN = True`), und verschiebe erst danach mit `Path.rename`. Eine Datei, die es am Ziel schon gibt, wird nicht überschrieben, sondern gemeldet. Die Zelle darunter legt den Ordner mit Beispieldateien an.

In [ ]:
import shutil

# Vorbereitung: legt den Beispielordner neu an. Gelöscht wird nur dieser Übungsordner.
downloads = Path("../daten/ausgabe/downloads")
shutil.rmtree(downloads, ignore_errors=True)
(downloads / "pdf").mkdir(parents=True)
(downloads / "alt").mkdir()
for name in [
    "rechnung_oktober.pdf",
    "vertrag.pdf",
    "urlaub_01.JPG",
    "urlaub_02.jpg",
    "einkaufsliste.txt",
    "haushaltsbuch.xlsx",
    "README",
]:
    (downloads / name).write_text(f"Inhalt von {name}\n", encoding="utf-8")
(downloads / "pdf" / "vertrag.pdf").write_text("ältere Fassung\n", encoding="utf-8")
(downloads / "alt" / "notiz.txt").write_text("alte Notiz\n", encoding="utf-8")
print(sorted(path.name for path in downloads.iterdir()))

In [ ]:
DRY_RUN = True  # erst prüfen, dann auf False setzen und erneut ausführen
downloads = Path("../daten/ausgabe/downloads")
for path in sorted(downloads.iterdir()):
    if not path.is_file():
        continue
    target = downloads / (path.suffix.lower().removeprefix(".") or "ohne_endung") / path.name
    if target.exists():
        print(f"übersprungen, existiert schon: {target}")
    elif DRY_RUN:
        print(f"würde verschieben: {path.name} -> {target.parent.name}/")
    else:
        target.parent.mkdir(exist_ok=True)
        path.rename(target)

**Übung 7 (Ausgaben je Empfänger, mittel, für Schnelle):** Erweitere deine Auswertung des Kontoauszugs aus Übung 5: Summiere die Ausgaben je Empfänger in einem Dictionary und gib sie absteigend sortiert aus. Erwartet in der ersten Zeile: `Hausverwaltung Meier 1970.00 €`.

Tipp: `by_payee.get(name, Decimal(0))` liefert 0, solange ein Empfänger noch fehlt; sortieren kannst du mit `sorted(by_payee.items(), key=lambda item: item[1], reverse=True)`.

In [ ]:
statement = Path("../daten/ausgabe/beispieldaten/kontoauszug.csv")
with open(statement, encoding="utf-8-sig", newline="") as file:
    rows = list(csv.DictReader(file, delimiter=";"))

by_payee = {}
for row in rows:
    if not row["Betrag"]:
        continue
    amount = Decimal(row["Betrag"].replace(".", "").replace(",", "."))
    if amount < 0:
        payee = row["Empfänger"]
        by_payee[payee] = by_payee.get(payee, Decimal(0)) - amount

for payee, spent in sorted(by_payee.items(), key=lambda item: item[1], reverse=True):
    print(f"{payee:<20} {spent:>8} €")

## Zusammenfassung

- Relative Pfade gelten ab dem Arbeitsordner, in Jupyter dem Ordner des Notebooks.
- `Path("ordner") / "datei.txt"` baut Pfade, `.exists()`, `.iterdir()`, `.mkdir()` helfen beim Umgang.
- Immer `with open(pfad, modus, encoding="utf-8") as file:`; zwei Dateien in einem `with`, durch Komma getrennt.
- Modus `"r"` lesen, `"w"` überschreiben, `"a"` anhängen. Kleine Dateien: `read_text()` und `write_text()`.
- CSV: `csv.DictReader` und `csv.DictWriter`, dabei `newline=""` angeben. Für Excel: `encoding="utf-8-sig"`.
- JSON: `json.load` / `json.dump(..., indent=2, ensure_ascii=False)`.